# Understanding blockchains from scratch

This notebook is a companion to the medium article [Understanding blockchains from scratch](https://medium.com/mitb-for-all/understanding-blockchains-from-scratch-f14c454af32f)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import hashlib
import json
import time

# Concept 1: Hash chaining makes history tamper-evident

A minimal, dependency-free blockchain to show **one** idea clearly:

each block stores the cryptographic hash of the block before it. If you
change any past block, either that block's own fingerprint no longer
matches its contents, or the next block's `previous_hash` pointer no
longer matches — so a validator can reject the chain.

**Not covered yet:** mining / proof-of-work, peer-to-peer networks, or
consensus (how many machines agree on one history). This section is only
the "chain" part of "blockchain."

## Anatomy of a block

A block is a labelled box: **data** (whatever you want to record) plus
metadata that ties it into the chain.

In the class below:

| Field | Role |
| --- | --- |
| `index` | Position in the chain (`0` = genesis) |
| `timestamp` | When this block was created |
| `data` | The payload (a log line here; transactions in a real coin) |
| `previous_hash` | Hash of the prior block — **the link** |
| `hash` | SHA-256 fingerprint of *this* block's contents |

**Hash properties that matter here:**

- **Deterministic** — same input always produces the same hash
- **Avalanche effect** — a tiny input change scrambles the output
- **One-way (preimage resistance)** — you cannot usefully reverse a hash
  to invent matching contents

The payload format is up to the application. The chain only cares that
whatever you put in the box gets fingerprinted and linked.

In [3]:
class Block:
    def __init__(self, index, data, previous_hash):
        self.index = index
        self.timestamp = time.time()
        self.data = data
        self.previous_hash = previous_hash
        self.hash = self.compute_hash()

    def compute_hash(self):
        # Turn this block's contents into one deterministic string...
        block_contents = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
        }, sort_keys=True)
        # ...then hash it. Same input -> same output. Always. Everywhere.
        return hashlib.sha256(block_contents.encode()).hexdigest()

    def __repr__(self):
        return (f"Block #{self.index}\n"
                f"  data:          {self.data}\n"
                f"  previous_hash: {self.previous_hash[:16]}...\n"
                f"  hash:          {self.hash[:16]}...\n")

## Anatomy of a chain

Think of the chain as an append-only timeline: each new block is a new
event that locks onto the tip by copying the previous block's `hash`
into its own `previous_hash`.

Every chain starts with a **genesis block** — the first event, which has
no real parent. Here we use a dummy `previous_hash` of sixty-four zeros.

`is_valid()` walks the list and enforces two rules:

1. **Integrity** — each block's stored `hash` still matches a fresh
   `compute_hash()` of its current contents
2. **Linkage** — each block's `previous_hash` equals the previous
   block's `hash`

> Honest claim for this section: hash linking makes tampering
> **detectable**. It does *not* yet make rewriting **expensive**.
> On a short toy chain, an attacker who controls this one copy can still
> rebuild a consistent history by recomputing hashes from the edit
> forward. Proof-of-work (Concept 2) is what turns that rebuild into
> real work. A real network also needs consensus so many nodes agree
> on one tip — that comes later.


In [4]:
class Blockchain:
    def __init__(self):
        # every chain needs a first block with no parent -- the "genesis block"
        self.chain = [Block(0, "Genesis Block", "0" * 64)]

    def add_block(self, data):
        previous_block = self.chain[-1]
        new_block = Block(len(self.chain), data, previous_block.hash)
        self.chain.append(new_block)

    def is_valid(self):
        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i - 1]

            # Rule 1: this block's stored hash must match what we get
            # from recomputing it right now.
            if current.hash != current.compute_hash():
                return False, f"Block #{current.index} was tampered with directly."

            # Rule 2: this block must correctly point to the block before it.
            if current.previous_hash != previous.hash:
                return False, f"Block #{current.index} is disconnected from Block #{previous.index}."

        return True, "Chain is valid."

    def print_chain(self):
        for block in self.chain:
            print(block)

In [5]:
print("=== Building the chain ===\n")
titus_chain = Blockchain()
titus_chain.add_block("Titus starts at the SMU MITB AI track in Aug 2021")
titus_chain.add_block("Titus graduates from MITB AI track in 2023")
titus_chain.add_block("Titus gets caught up in the genAI wave because of ChatGPT's launch")

titus_chain.print_chain()

valid, msg = titus_chain.is_valid()
print(f"Chain valid? {valid} -- {msg}\n")


=== Building the chain ===

Block #0
  data:          Genesis Block
  previous_hash: 0000000000000000...
  hash:          9d6dbf9397d521fe...

Block #1
  data:          Titus starts at the SMU MITB AI track in Aug 2021
  previous_hash: 9d6dbf9397d521fe...
  hash:          fdc17264b49269f2...

Block #2
  data:          Titus graduates from MITB AI track in 2023
  previous_hash: fdc17264b49269f2...
  hash:          9db324db6664fc1b...

Block #3
  data:          Titus gets caught up in the genAI wave because of ChatGPT's launch
  previous_hash: 9db324db6664fc1b...
  hash:          2350b37714986846...

Chain valid? True -- Chain is valid.



## Attack demo: rewrite a past event

A normal database lets you `UPDATE` an old row and move on. Here we mutate
Block #1 in place and ask: does the chain still look valid?

Two attempts:

1. **Sloppy** — change the data, leave the old stored hash
2. **Clever** — also recompute Block #1's hash to hide the edit

Watch which validation rule catches each attempt.

In [6]:
"""Attempt 1: sloppy tamper.

Change Block #1's data but leave its stored hash alone.
is_valid() Rule 1 catches this: stored hash no longer matches
what compute_hash() produces from the (now-altered) contents.
"""
print("=== Attempt 1: sloppy tamper -- edit the data, leave the old hash ===\n")

# Mutate history in place; do NOT refresh the block's hash.
titus_chain.chain[1].data = "Titus graduates from the MITB FinTech track"

# Expect: False -- Block #1 was tampered with directly.
valid, msg = titus_chain.is_valid()
print(f"Chain valid? {valid} -- {msg}\n")

=== Attempt 1: sloppy tamper -- edit the data, leave the old hash ===

Chain valid? False -- Block #1 was tampered with directly.



In [7]:
print("=== Attempt 2: clever tamper -- also recompute Block #1's own hash to cover tracks ===\n")
titus_chain.chain[1].hash = titus_chain.chain[1].compute_hash()

valid, msg = titus_chain.is_valid()
print(f"Chain valid? {valid} -- {msg}")

=== Attempt 2: clever tamper -- also recompute Block #1's own hash to cover tracks ===

Chain valid? False -- Block #2 is disconnected from Block #1.


### What the two attempts showed

**Attempt 1 (sloppy):** change data, leave the old hash.  
Rule 1 fails immediately — the stored fingerprint no longer matches the
contents ("tampered with directly").

**Attempt 2 (clever):** recompute Block #1's hash after the edit.  
Rule 1 now passes for Block #1, but Block #2 still stores the *old*
parent hash. Rule 2 fails: Block #2 is **disconnected** from Block #1.

The chain is invalid either way — and that is the point. Validation
does not only inspect blocks in isolation; it also checks that every
`previous_hash` still points at the real previous block.

To forge a *consistent* chain in this toy model (still no mining), an
attacker must cascade the repair:

1. Change the target block's data
2. Recompute that block's hash
3. Update the next block's `previous_hash` to the new hash
4. Recompute *that* block's hash (`previous_hash` is part of its input)
5. Repeat for every block after the edit

You cannot change one middle block and leave the rest alone. On a short
notebook chain that cascade is still cheap CPU — which is why Concept 2
adds proof-of-work: after each content change, you must also **re-mine**.

# Concept 2: Proof of work — hard to produce, easy to check

Same hash linking as Concept 1, but **writing** a block now costs real
CPU time. We require each block's hash to start with `difficulty`
leading zeros. Finding such a hash means guessing a **nonce** over and
over until the hash accidentally fits. That search is **mining**.

Mining and verifying are not the same operation:

| | Mining | Verifying |
| --- | --- | --- |
| What you do | Try nonce = 0, 1, 2, … and re-hash | Hash once with the given nonce |
| Cost | Often hundreds of thousands of tries here | One hash — essentially free |
| Who | Whoever wants to append a block | Anyone auditing the chain |

> **Sudoku analogy:** finishing a hard puzzle takes real effort; checking
> a finished grid is quick. That asymmetry — hard to produce, trivial to
> check — is the shape of proof-of-work. One miner sweats; many verifiers
> can audit for nearly free.

**How this notebook models difficulty:** `difficulty = 5` means the hash
must start with `00000`. Each extra leading zero multiplies expected work
by about **16×** (one more hex digit). Bitcoin compares against a numeric
target rather than counting literal zeros, but the teaching idea is the
same: rarer acceptable hashes ⇒ more work on average.

`is_valid()` gains a third rule on top of integrity + linkage:

3. **Proof of work** — the hash still meets the leading-zeros target
   (someone actually did the search; you cannot just recompute a normal hash)

In [8]:
class Block:
    """A proof-of-work block.

    Same linking idea as before (previous_hash), but writing a valid
    hash now requires finding a nonce such that the hash starts with
    `difficulty` leading zeros. That search is mining.
    """

    def __init__(self, index, data, previous_hash, difficulty=5):
        """Build a block and mine it immediately.

        Args:
            index: Position in the chain (0 = genesis).
            data: Payload stored in this block.
            previous_hash: Hash of the prior block (links the chain).
            difficulty: Required number of leading zeros in the hash.
        """
        self.index = index
        self.timestamp = time.time()
        self.data = data
        self.previous_hash = previous_hash
        self.difficulty = difficulty   # how many leading zeros the hash must have
        self.nonce = 0                 # counter we brute-force over until the hash fits
        self.hash = self.mine()        # mining sets nonce + returns a valid hash

    def compute_hash(self):
        """SHA-256 of this block's contents (including the current nonce).

        Same inputs always produce the same hash. Changing data, previous_hash,
        or nonce changes the hash — which is why tampering forces re-mining.
        """
        block_contents = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce,          # nonce is part of what gets hashed
        }, sort_keys=True)
        return hashlib.sha256(block_contents.encode()).hexdigest()

    def mine(self):
        """Brute-force nonce until compute_hash() meets the difficulty target.

        Target example: difficulty=5 means the hash must start with "00000".
        Higher difficulty → exponentially more attempts on average.
        """
        target = "0" * self.difficulty
        start = time.time()
        attempts = 0
        candidate_hash = self.compute_hash()

        # Keep bumping nonce and re-hashing until we hit the leading-zeros target.
        while not candidate_hash.startswith(target):
            self.nonce += 1
            attempts += 1
            candidate_hash = self.compute_hash()

        elapsed = time.time() - start
        print(f"  mined Block #{self.index} in {attempts:,} attempts, {elapsed:.2f}s "
              f"(nonce={self.nonce}, hash={candidate_hash[:16]}...)")
        return candidate_hash

    def __repr__(self):
        """Readable summary for notebook printing."""
        return (f"Block #{self.index}\n"
                f"  data:          {self.data}\n"
                f"  nonce:         {self.nonce}\n"
                f"  previous_hash: {self.previous_hash[:16]}...\n"
                f"  hash:          {self.hash[:16]}...\n")

In [ ]:
class Blockchain:
    """A proof-of-work chain of Block objects.

    Extends the earlier linking rules with a third check: every block's
    hash must still meet the difficulty target (leading zeros). That means
    rewriting history requires re-mining every block after the change.
    """

    def __init__(self, difficulty=5):
        """Start the chain with a mined genesis block.

        Args:
            difficulty: Leading-zero requirement shared by every new block.
        """
        self.difficulty = difficulty
        # Genesis has no real parent; use a fixed dummy previous_hash.
        self.chain = [Block(0, "Genesis Block", "0" * 64, difficulty)]

    def add_block(self, data):
        """Append a new mined block linked to the current tip of the chain.

        Creating Block(...) runs mine() inside __init__, so this call blocks
        until a valid nonce is found for the given difficulty.
        """
        previous_block = self.chain[-1]
        new_block = Block(len(self.chain), data, previous_block.hash, self.difficulty)
        # Validation code here. A few other computers validate this block fulfills the 
        # difficulty level before it is allowed to append.
        self.chain.append(new_block)

    def is_valid(self):
        """Validate integrity, linkage, and proof-of-work for every block.

        Rules checked (from index 1 onward):
          1. Stored hash matches recomputed hash (no silent data edits).
          2. previous_hash matches the prior block's hash (chain is linked).
          3. Hash still starts with the required number of zeros (work was done).

        Returns:
            (True, message) if valid; (False, reason) on the first failure.
        """
        target = "0" * self.difficulty
        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i - 1]

            # Rule 1: contents weren't altered after mining.
            if current.hash != current.compute_hash():
                return False, f"Block #{current.index} was tampered with directly."

            # Rule 2: this block still points at the real previous block.
            if current.previous_hash != previous.hash:
                return False, f"Block #{current.index} is disconnected from Block #{previous.index}."

            # Rule 3: the hash still proves work at the current difficulty.
            if not current.hash.startswith(target):
                return False, f"Block #{current.index} doesn't satisfy the difficulty target -- no real work was done."

        return True, "Chain is valid."

## Mine a short chain

Run the cell below at `difficulty=5`. Watch `mine()` print **attempts**,
**seconds**, **nonce**, and a hash that starts with `00000` for every
block — including genesis, which is mined inside `Blockchain.__init__`.

That per-block log *is* the teaching signal: appending history now costs
guesses, while `is_valid()` still finishes in a blink.

In [12]:
print("=== Mining a chain at difficulty 5 (hash must start with '00000') ===\n")
chain = Blockchain(difficulty=5)
chain.add_block("Titus starts at the SMU MITB AI track in Aug 2021")
chain.add_block("Titus graduates from MITB AI track in 2023")

valid, msg = chain.is_valid()
print(f"\nChain valid? {valid} -- {msg}\n")

=== Mining a chain at difficulty 5 (hash must start with '00000') ===

  mined Block #0 in 874,881 attempts, 1.91s (nonce=874881, hash=00000ccb975e205b...)
  mined Block #1 in 425,329 attempts, 0.95s (nonce=425329, hash=0000078e8891e7b6...)
  mined Block #2 in 130,069 attempts, 0.29s (nonce=130069, hash=000002fb925bd824...)

Chain valid? True -- Chain is valid.



### Reading the mining output

What you just saw is proof-of-work in "easy classroom" mode: each block
took on the order of \(10^5\)–\(10^6\) hashes. Bump difficulty by one and
expect roughly **16×** more work on average.

**How this relates to Bitcoin (accurate enough for 101):**

- Real Bitcoin does **not** use a fixed "19 leading zeros" knob. It uses a
  numeric **target**; lower target ⇒ harder. Difficulty is adjusted about
  every **2016 blocks** (~two weeks) so average block time stays near
  **10 minutes**, even as global hashpower changes — a thermostat, not a
  constant.
- Mining is a parallel race: many machines guess nonces at once; the first
  valid proof wins the right to propose the next block (plus the reward).
- This notebook is still a **single local chain**. PoW makes rewriting
  *your* history expensive. A real network also needs **consensus** (many
  nodes, one agreed tip). Controlling enough hashpower to outpace honest
  miners is the classic **51% attack** — out of scope here, but worth
  naming so PoW is not mistaken for "unhackable by definition."

## Attack demo: edit data, re-hash, skip re-mining

Concept 1's "clever" attacker recomputed hashes and could, with enough
cascading edits, rebuild a valid-looking chain for free.

With proof-of-work, that shortcut dies. The cell below changes Block #1's
data and calls `compute_hash()` once — **no** nonce search. Rule 1 may
pass (stored hash matches contents), and Rule 2 may still hold if we only
touch this block's data/hash fields carefully, but **Rule 3** fails:
the new hash almost certainly does not start with the required zeros.

Lesson: after PoW, forging history means re-mining the tampered block
*and* every block after it at the current difficulty.

In [13]:
print("=== Attacker fakes a block's DATA and re-hashes it, but skips re-mining ===\n")
fake_block = chain.chain[1]
fake_block.data = "Titus actually works for a rival bank"
fake_block.hash = fake_block.compute_hash()   # recomputed, but never re-mined!

valid, msg = chain.is_valid()
print(f"Chain valid? {valid} -- {msg}")

=== Attacker fakes a block's DATA and re-hashes it, but skips re-mining ===

Chain valid? False -- Block #1 doesn't satisfy the difficulty target -- no real work was done.


### Why you cannot "just type" a hash that starts with zeros

A hash is not a field you fill in by hand. It is the **output** of
SHA-256 on the block contents (including the nonce). Commit to the
input and the fingerprint falls out instantly; there is no partial credit
and no "getting warmer" toward a nicer-looking hash.

Two properties make cheating fail:

- **Preimage resistance** — given a desired hash (e.g. one with many
  leading zeros), finding an input that produces it is computationally
  infeasible except by guessing
- **Avalanche effect** — change one character of the input and the
  output scrambles unpredictably

So the only practical way to satisfy the difficulty target is the
brute-force loop in `mine()`: bump the nonce, hash, check, repeat.
Verifiers do not redo that search — they hash once and check the prefix.

## Takeaways (blockchain 101 so far)

1. **Blocks** fingerprint their contents with a cryptographic hash.
2. **Chains** link blocks with `previous_hash`, so edits ripple forward.
3. **Validation** checks integrity + linkage (+ proof-of-work once mining exists).
4. **Proof-of-work** makes appending (and rewriting) expensive, while verifying stays cheap.
5. **Still not covered:** digital signatures / wallets, Merkle trees of transactions, peer-to-peer gossip, forks, or consensus rules for choosing among competing tips.

Hash chains + PoW explain *why forging a long history is costly*. Consensus explains *why the network converges on one history*. Both matter; this notebook focused on the first half.